Test CatBoost Model on real data to see what it produces for an output

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from core import Config

config = Config()
model_path = ("2_imputed_thresh_10_log_model")
TARGET_COL = "TR.UpstreamScope3PurchasedGoodsAndServices"
verification_path = "imputed_thresh_10_verification-r.csv"
ID_COLS = ["Instrument"]
FEATURE_COLS_35 = ['Instrument', 'Date', 'TR.TotalAssetsActual', 'TR.F.PPENetTot',
       'TR.InventoryActValue', 'TR.CO2IndirectScope2', 'TR.F.OpExpn',
       'TR.F.GrossProfMarg', 'TR.NumberofEmployees', 'TR.EnergyUseTotal',
       'TR.CurrentAssetsActValue', 'TR.CapexActValue',
       'TR.F.GrossProfIndPropTot', 'TR.GPMActValue',
       'TR.CurrentLiabilitiesActValue', 'TR.GrossIncomeActValue',
       'TR.CO2DirectScope1', 'TR.F.AcctRcvblTurnover',
       'TR.UpstreamScope3PurchasedGoodsAndServices',
       'TR.TakebackRecyclingInitiatives', 'TR.RenewEnergyUse',
       'TR.PolicyEmissions', 'TR.PolicyEnergyEfficiency',
       'TR.PolicyEnvSupplyChain', 'TR.PolicyFairTrade',
       'TR.PolicySustainablePackaging', 'TR.F.InvntTurnover',
       'TR.GICSSectorCode', 'TR.HQCountryCode']
if "thresh_50" in model_path:
    FEATURE_COLS = [

    ]
else:
    FEATURE_COLS = [  # exactly the same feature list used during training
        'Date', 'TR.TotalAssetsActual', 'TR.F.PPENetTot',
        'TR.CO2IndirectScope2', 'TR.F.OpExpn', 'TR.NumberofEmployees',
        'TR.CO2DirectScope1', 'TR.TakebackRecyclingInitiatives', 'TR.RenewEnergyUse',
        'TR.PolicyEmissions', 'TR.PolicyEnergyEfficiency',
        'TR.PolicyEnvSupplyChain', 'TR.PolicyFairTrade',
        'TR.PolicySustainablePackaging', 'TR.GICSSectorCode',
        'TR.HQCountryCode'
    ]
# FEATURE_COLS = FEATURE_COLS_35
CAT_FEATURES = [
    'TR.GICSSectorCode',
    'TR.HQCountryCode'
]


def load_model(path: str) -> CatBoostRegressor:
    model = CatBoostRegressor()
    model.load_model(f"{config.results_dir}/model/{path}.bin")
    return model


def prepare_features(df_input: pd.DataFrame) -> pd.DataFrame:
    df = df_input.copy()

    missing_features = [c for c in FEATURE_COLS if c not in df.columns]
    if missing_features:
        raise ValueError(f"Missing required feature columns: {missing_features}")

    X = df[FEATURE_COLS].copy()

    for col in CAT_FEATURES:
        X[col] = X[col].astype("string").fillna("missing")

    num_cols = [c for c in FEATURE_COLS if c not in CAT_FEATURES]
    for col in num_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    return X


def predict_with_actual(model, df_input: pd.DataFrame) -> pd.DataFrame:
    df = df_input.copy()
    X = prepare_features(df)

    pred_pool = Pool(
        data=X,
        cat_features=CAT_FEATURES
    )

    y_pred = model.predict(pred_pool)
    # convert log back to number
    y_pred = pd.Series(y_pred).apply(lambda x: np.exp(x))

    result_cols = [c for c in ID_COLS if c in df.columns]
    result = df[result_cols].copy() if result_cols else pd.DataFrame(index=df.index)

    if TARGET_COL in df.columns:
        result["actual"] = pd.to_numeric(df[TARGET_COL], errors="coerce")

    result["predicted"] = y_pred

    if "actual" in result.columns:
        result["error"] = result["predicted"] - result["actual"]
        result["abs_error"] = result["error"].abs()
        result["abs_error_pct"] = result["abs_error"] / result["actual"] * 100

    return result


cb_model = load_model(model_path)
df_test = pd.read_csv(config.training_dir / verification_path, index_col=0)
result = predict_with_actual(cb_model, df_test)
result.to_csv(f"{config.results_dir}/verifications_predictions_{model_path}.csv")
result.head()

,Instrument,actual,predicted,error,abs_error,abs_error_pct
0,SCATC.OL,1630.0,1707.645954,77.645954,77.645954,4.763555
1,SCATC.OL,2036.0,1934.828184,-101.171816,101.171816,4.969146
2,SCATC.OL,2643.0,2204.497803,-438.502197,438.502197,16.591078
3,SCATC.OL,6990.0,2301.996558,-4688.003442,4688.003442,67.067288
4,SCATC.OL,14272.0,3987.660700,-10284.339300,10284.339300,72.059552
